### Bibliotecas Necessárias

In [1]:
import os
import boto3
import pymysql
import cryptography
from dotenv import load_dotenv

load_dotenv()

AWS_REGION = os.getenv('AWS_REGION')
DB_INSTANCE_ID = os.getenv('DB_INSTANCE_ID')
DB_HOST = os.getenv('DB_HOST')
DB_NAME = os.getenv('DB_NAME')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')

### Provisionamento

In [3]:
rds = boto3.client('rds', region_name=AWS_REGION)

response = rds.create_db_instance(
    DBInstanceIdentifier=DB_INSTANCE_ID,
    AllocatedStorage=20,
    DBInstanceClass='db.t3.micro',
    Engine='mysql',
    MasterUsername=DB_USER,
    MasterUserPassword=DB_PASSWORD,
    PubliclyAccessible=True
)

print("Instância criada")

Instância criada


### Liberando a Porta do MySQL

In [10]:
rds = boto3.client('rds', region_name=AWS_REGION)
ec2 = boto3.client('ec2', region_name=AWS_REGION)

instancia = rds.describe_db_instances(DBInstanceIdentifier=DB_INSTANCE_ID)
sg_id = instancia['DBInstances'][0]['VpcSecurityGroups'][0]['VpcSecurityGroupId']

try:
    ec2.authorize_security_group_ingress(
        GroupId=sg_id,
        IpProtocol='tcp',
        FromPort=3306,
        ToPort=3306,
        CidrIp='0.0.0.0/0'
    )
    print(f"Porta 3306 liberada no Security Group {sg_id}.")
except Exception as e:
    print(f"Resultado: {e}")

Resultado: An error occurred (InvalidPermission.Duplicate) when calling the AuthorizeSecurityGroupIngress operation: the specified rule "peer: 0.0.0.0/0, TCP, from port: 3306, to port: 3306, ALLOW" already exists


### Load de Dados

In [11]:
from pathlib import Path
from pymysql.constants import CLIENT

SQL_PATH = Path('../../data/mysqlsampledatabase.sql').resolve()

conexao = pymysql.connect(
    host=DB_HOST,
    user=DB_USER,
    password=DB_PASSWORD,
    client_flag=CLIENT.MULTI_STATEMENTS
)
cursor = conexao.cursor()

cursor.execute(f"create database if not exists {DB_NAME}")
cursor.execute(f"use {DB_NAME}")

with open(SQL_PATH, 'r', encoding='utf-8') as arquivo:
    sql = arquivo.read()

cursor.execute(sql)

conexao.commit()
conexao.close()
print('Banco de dados criado')

Banco de dados criado


### Validação

In [12]:
conexao = pymysql.connect(
    host=DB_HOST,
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME
)
cursor = conexao.cursor()

cursor.execute(f"select table_name from information_schema.tables where table_schema = '{DB_NAME}'")
tabelas = cursor.fetchall()

for tabela in tabelas:
    nome_tabela = tabela[0]
    cursor.execute(f"select count(*) from `{nome_tabela}`")
    qtd = cursor.fetchone()[0]
    print(f"Tabela: {nome_tabela} | Total de Registros: {qtd}")

conexao.close()

Tabela: customers | Total de Registros: 122
Tabela: employees | Total de Registros: 23
Tabela: offices | Total de Registros: 7
Tabela: orderdetails | Total de Registros: 2996
Tabela: orders | Total de Registros: 326
Tabela: payments | Total de Registros: 273
Tabela: productlines | Total de Registros: 7
Tabela: products | Total de Registros: 110


### Deletar Instância


In [5]:
rds = boto3.client('rds', region_name=AWS_REGION)

rds.delete_db_instance(
    DBInstanceIdentifier=DB_INSTANCE_ID,
    SkipFinalSnapshot=True
)

{'DBInstance': {'DBInstanceIdentifier': 'db-classicmodels',
  'DBInstanceClass': 'db.t3.micro',
  'Engine': 'mysql',
  'DBInstanceStatus': 'deleting',
  'MasterUsername': 'admin',
  'Endpoint': {'Address': 'db-classicmodels.cxttwhozt6l7.us-east-1.rds.amazonaws.com',
   'Port': 3306,
   'HostedZoneId': 'Z2R2ITUGPM61AM'},
  'AllocatedStorage': 20,
  'InstanceCreateTime': datetime.datetime(2026, 4, 4, 21, 50, 47, 912000, tzinfo=tzutc()),
  'PreferredBackupWindow': '05:17-05:47',
  'BackupRetentionPeriod': 1,
  'DBSecurityGroups': [],
  'VpcSecurityGroups': [{'VpcSecurityGroupId': 'sg-083331300bc0f11d7',
    'Status': 'active'}],
  'DBParameterGroups': [{'DBParameterGroupName': 'default.mysql8.4',
    'ParameterApplyStatus': 'in-sync'}],
  'AvailabilityZone': 'us-east-1b',
  'DBSubnetGroup': {'DBSubnetGroupName': 'default',
   'DBSubnetGroupDescription': 'default',
   'VpcId': 'vpc-0976f866f488c5326',
   'SubnetGroupStatus': 'Complete',
   'Subnets': [{'SubnetIdentifier': 'subnet-060a5b5a5